# 05 · Ensinando a máquina ao vivo — **EPI na obra**

O ponto alto da palestra: a plateia vê a IA **aprendendo na frente dela**, com
fotos que nasceram nesta sala, há dois minutos.

| passo | o que acontece | tempo |
|---|---|---|
| 0 | o modelo pronto **não sabe** o que é capacete | 40 s |
| 1 | **coletar** com a câmera: com EPI e sem EPI | 2 min |
| 2 | **separar** em duas pastas (o nome da pasta é o gabarito) | 20 s |
| 3 | **treinar**, narrando a curva enquanto ela desenha | 2 min |
| 4 | **provar** com fotos que ele nunca viu | 40 s |
| 5 | **rodar ao vivo** na câmera, apontando para a plateia | 1 min |

> Fala de abertura: *"eu não vou escrever nenhuma regra sobre capacete.
> Nenhum 'se for amarelo e redondo'. Eu vou mostrar exemplos, e é exatamente
> assim que a coisa toda funciona, do ChatGPT a isso aqui."*

**Leve na mala:** um capacete e um óculos de proteção. Se esquecer, qualquer
par de objetos vestíveis serve, e a demonstração é a mesma.

In [ ]:
# ── 1. instala a biblioteca e monta o Google Drive ──
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics, torch, os, glob
ultralytics.checks()
print("GPU disponivel:", torch.cuda.is_available())

# ── 2. a raiz de tudo, e a conferência de que ela é REAL ──────────────
#
# ARMADILHA que já custou uma sessão: a linha acima cria a variável
# `drive` (minúscula), que é o MÓDULO do Colab. Se algum caminho for
# escrito com `drive` em vez de `DRIVE`, o Python aceita numa boa e
# monta um caminho como
#     <module 'google.colab.drive' from '/usr/local/...'>/04-garrafas
# O código roda, cria pastas, exporta arquivos — tudo no disco
# temporário do Colab, que evapora quando a sessão encerra. Nada disso
# chega ao seu Drive, e não há erro nenhum na tela.
#
# A conferência abaixo transforma esse silêncio num aviso imediato.

DRIVE = "/content/drive/MyDrive/PALESTRA-IA"

if not DRIVE.startswith("/content/drive/"):
    raise SystemExit(
        "DRIVE aponta para fora do Google Drive: " + repr(DRIVE) + "\n"
        "Provavelmente algum caminho usou `drive` (o módulo) em vez de `DRIVE`.")
if not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("O Drive não montou. Rode esta célula de novo e autorize o acesso.")

os.makedirs(DRIVE, exist_ok=True)
print("raiz no Drive:", DRIVE)
print("existe de verdade:", os.path.isdir(DRIVE))

In [ ]:
# ── ajuste de PALCO: tudo grande, porque a sala enxerga de 6 a 10 m ──
import matplotlib
matplotlib.rcParams.update({
    "figure.figsize": (16, 9),
    "figure.dpi": 110,
    "font.size": 22,
    "axes.titlesize": 30,
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 22,
    "axes.grid": True,
    "grid.alpha": .25,
    "axes.facecolor": "#0d1117",
    "figure.facecolor": "#0d1117",
    "text.color": "#e6edf3",
    "axes.labelcolor": "#e6edf3",
    "xtick.color": "#e6edf3",
    "ytick.color": "#e6edf3",
    "axes.edgecolor": "#30363d",
    "axes.titlecolor": "#3fe0a8",
})
VERDE, VERMELHO, CINZA = "#3fe0a8", "#ff5c5c", "#7d8590"
# DRIVE não é redefinido aqui de propósito: quem define é a célula de
# setup, e uma variável de caminho com duas origens é como se perde a
# noção de onde os arquivos foram parar.
print("palco configurado")

## Passo 0 · o modelo pronto não sabe o que você precisa

Tire uma foto **com o capacete**, e repare no que o modelo genérico enxerga:
uma pessoa. O capacete não existe para ele.

> *"Ele não é burro. Ele nunca viu um. Ninguém mostrou."*

In [ ]:
# ═══════════════════════════════════════════════════════════════
# UMA FOTO SO — o plano B que nunca falha
#
# Serve para dois momentos:
#   · mostrar o modelo generico ERRANDO, antes de treinar qualquer coisa
#   · rodar a demo quando a camera continua nao abrir na maquina do evento
#
# Nao depende do laco de video: tira, salva e devolve o caminho.
# ═══════════════════════════════════════════════════════════════
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode


def foto(nome="/content/foto.jpg", qualidade=0.92, rotulo="tirar a foto"):
    display(Javascript("""
      async function tirarFoto(rotulo, qualidade){
        const div = document.createElement('div');
        div.style.cssText = 'border:2px solid #3fe0a8;border-radius:12px;'
          + 'padding:8px;max-width:760px;background:#0b1418';
        const botao = document.createElement('button');
        botao.textContent = rotulo;
        botao.style.cssText = 'width:100%;cursor:pointer;background:#3fe0a8;'
          + 'color:#08151a;font-weight:700;font-size:17px;border:none;'
          + 'border-radius:8px;padding:12px;margin-bottom:8px';
        const video = document.createElement('video');
        video.style.cssText = 'width:100%;display:block;border-radius:8px';
        video.setAttribute('playsinline','');
        div.appendChild(botao); div.appendChild(video);
        document.body.appendChild(div);

        const stream = await navigator.mediaDevices.getUserMedia({video:true});
        video.srcObject = stream; await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((r) => botao.onclick = r);

        const tela = document.createElement('canvas');
        tela.width = video.videoWidth; tela.height = video.videoHeight;
        tela.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return tela.toDataURL('image/jpeg', qualidade);
      }
    """))
    dados = eval_js('tirarFoto("{}", {})'.format(rotulo, qualidade))
    with open(nome, "wb") as f:
        f.write(b64decode(dados.split(",")[1]))
    print("foto salva em", nome)
    return nome

In [ ]:
import cv2, matplotlib.pyplot as plt

generico = YOLO(f"{DRIVE}/00-pesos/yolo11n.pt")

antes = foto("/content/antes-do-treino.jpg", rotulo="clique com o EPI vestido")
r = generico.predict(antes, conf=.30, verbose=False)[0]

achou = sorted({generico.names[int(c)] for c in r.boxes.cls}) if len(r.boxes) else []
print("o que o modelo pronto enxerga:", achou or "nada acima da confianca")
print("existe alguma classe de EPI? ",
      any(("helmet" in a) or ("hat" in a) or ("glass" in a) for a in achou))
print()
print("ele conhece 80 categorias. Capacete de obra nao e uma delas,")
print("e o defeito do SEU processo tambem nao vai ser.")

plt.figure()
plt.imshow(cv2.cvtColor(r.plot(line_width=4), cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.title("Modelo genérico: vê a pessoa, ignora o EPI")
plt.tight_layout(); plt.show()

## Passo 1 · o estúdio de coleta

Abra a câmera e grave os dois lados. **Deixe a rajada ligada**: um clique grava
doze fotos em pouco mais de dois segundos, e como você se mexe um pouco entre
elas, o material sai variado. É disso que o treino precisa.

**Como coletar bem, em dois minutos:**

| faça | por quê |
|---|---|
| vire o rosto, aproxime, afaste | o modelo aprende o objeto, não a sua pose |
| mude de lugar na sala | ele aprende o capacete, não o fundo da parede |
| chame alguém da plateia | duas pessoas diferentes valem mais que 200 fotos suas |
| colete os dois lados no mesmo lugar | senão ele aprende "parede clara = com EPI" |

Aquele último item é o erro clássico, e vale contar em voz alta: se todas as
fotos com EPI forem no palco e as sem EPI forem no fundo da sala, **o modelo
aprende o cenário**, não o capacete. Ele vai acertar no teste e falhar na obra.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ESTÚDIO DE COLETA — a câmera vira dataset, com dois cliques
#
# Como funciona no palco:
#   1. rode esta célula e a de baixo: a câmera abre
#   2. apareça na frente dela do jeito da PRIMEIRA classe
#   3. clique no botão daquela classe (ou aperte a tecla dele)
#   4. troque de estado e faça o outro lado
#
# A RAJADA é o que torna isso viável em dois minutos: um clique grava
# doze fotos em pouco mais de dois segundos, e como você se mexe um
# pouco entre elas, o material sai variado, que é exatamente o que o
# treino precisa. Dataset com cem fotos iguais não ensina nada.
# ═══════════════════════════════════════════════════════════════
from IPython.display import display, HTML, JSON
from google.colab import output as _saida_colab
from base64 import b64decode
import os, glob, json, time, random, shutil

_PALETA = ["#3fe0a8", "#ff5c7a", "#7cc4ff", "#ffb84d"]


class Estudio:
    """Coleta imagens da webcam, já separadas por classe."""

    def __init__(self, raiz, classes, meta=60):
        if not (2 <= len(classes) <= 4):
            raise ValueError("use de 2 a 4 classes — acima disso o painel "
                             "vira menu e o palco fica confuso")
        self.raiz = raiz
        self.classes = [str(c).strip().lower().replace(" ", "_") for c in classes]
        self.rotulos = [str(c).strip() for c in classes]
        self.meta = int(meta)
        self.bruto = os.path.join(raiz, "_bruto")
        for c in self.classes:
            os.makedirs(os.path.join(self.bruto, c), exist_ok=True)

        _saida_colab.register_callback("estudio.salvar", self._salvar)
        _saida_colab.register_callback("estudio.contar", self._contar)

    # ── disco ────────────────────────────────────────────────────
    def _pasta(self, classe):
        return os.path.join(self.bruto, classe)

    def arquivos(self, classe):
        return sorted(glob.glob(os.path.join(self._pasta(classe), "*.jpg")))

    def contagem(self):
        return {c: len(self.arquivos(c)) for c in self.classes}

    # ── callbacks chamados pelo navegador ────────────────────────
    def _contar(self):
        return JSON({"contagem": self.contagem(), "meta": self.meta})

    def _salvar(self, classe, dataurl):
        classe = str(classe)
        if classe not in self.classes:
            return JSON({"erro": "classe desconhecida: " + classe})
        try:
            dados = b64decode(dataurl.split(",", 1)[1])
        except Exception as e:
            return JSON({"erro": "imagem inválida: %s" % e})
        # carimbo em milissegundos: sem isso, a rajada se sobrescreve
        nome = "%s_%013d.jpg" % (classe, int(time.time() * 1000))
        with open(os.path.join(self._pasta(classe), nome), "wb") as f:
            f.write(dados)
        return JSON({"contagem": self.contagem(), "meta": self.meta})

    # ── o painel ─────────────────────────────────────────────────
    def abrir(self, largura=640, altura=480):
        cfg = {
            "classes": [
                {"id": c, "rotulo": r, "cor": _PALETA[i % len(_PALETA)],
                 "tecla": str(i + 1)}
                for i, (c, r) in enumerate(zip(self.classes, self.rotulos))
            ],
            "meta": self.meta,
            "larg": int(largura),
            "alt": int(altura),
            "inicial": self.contagem(),
        }
        display(HTML(_PAINEL_HTML.replace("__CFG__", json.dumps(cfg))))

    # ── relatório de texto, para conferir sem olhar o painel ─────
    def resumo(self):
        c = self.contagem()
        print("material coletado em", self.bruto)
        for k in self.classes:
            barra = "#" * int(30 * min(1.0, c[k] / max(1, self.meta)))
            print("  %-14s %4d  %s" % (k, c[k], barra))
        falta = [k for k in self.classes if c[k] < self.meta]
        if falta:
            print("\nainda abaixo da meta de %d: %s" % (self.meta, ", ".join(falta)))
        else:
            print("\ntodas as classes na meta. Pode preparar o dataset.")
        return c

    def limpar(self, confirmar=False):
        """Apaga TODO o material coletado. Precisa de confirmar=True."""
        if not confirmar:
            print("nada foi apagado. Chame limpar(confirmar=True) se for isso mesmo.")
            return
        shutil.rmtree(self.bruto, ignore_errors=True)
        for c in self.classes:
            os.makedirs(self._pasta(c), exist_ok=True)
        print("material apagado.")


_PAINEL_HTML = r"""
<div id="estudio-raiz">
<style>
  #estudio-raiz{
    --fg:#e9f2ef; --fg2:#9db3bc; --bg:#0b1418; --bg2:#122129;
    --linha:rgba(157,179,188,.22);
    font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",system-ui,sans-serif;
    color:var(--fg); background:var(--bg);
    border:1px solid var(--linha); border-radius:14px;
    padding:16px; max-width:1080px;
  }
  #estudio-raiz *{box-sizing:border-box;}
  #estudio-raiz .hdr{display:flex;align-items:baseline;gap:12px;flex-wrap:wrap;
    padding-bottom:12px;margin-bottom:14px;border-bottom:1px solid var(--linha);}
  #estudio-raiz .tit{font-size:19px;font-weight:700;letter-spacing:.14em;text-transform:uppercase;}
  #estudio-raiz .sub{font-size:14px;color:var(--fg2);}
  #estudio-raiz .stat{margin-left:auto;font-size:13px;color:var(--fg2);
    font-family:ui-monospace,SFMono-Regular,Menlo,monospace;}
  #estudio-raiz .grid{display:grid;grid-template-columns:1.25fr 1fr;gap:16px;}
  @media (max-width:820px){#estudio-raiz .grid{grid-template-columns:1fr;}}

  #estudio-raiz .cam{position:relative;background:#000;border-radius:12px;overflow:hidden;
    border:1px solid var(--linha);min-height:220px;}
  #estudio-raiz video{width:100%;display:block;transform:scaleX(-1);}
  #estudio-raiz .flash{position:absolute;inset:0;background:#fff;opacity:0;pointer-events:none;
    transition:opacity .18s ease;}
  #estudio-raiz .flash.on{opacity:.72;transition:none;}
  #estudio-raiz .hud{position:absolute;left:10px;bottom:10px;padding:6px 12px;border-radius:8px;
    background:rgba(0,0,0,.62);font-size:14px;font-family:ui-monospace,Menlo,monospace;}
  #estudio-raiz .regressiva{position:absolute;inset:0;display:none;align-items:center;
    justify-content:center;font-size:112px;font-weight:800;color:#fff;
    text-shadow:0 4px 30px rgba(0,0,0,.8);background:rgba(0,0,0,.25);}
  #estudio-raiz .regressiva.on{display:flex;}

  #estudio-raiz .ctrl{display:flex;flex-direction:column;gap:12px;}
  #estudio-raiz .lbl{font-size:11px;letter-spacing:.2em;text-transform:uppercase;color:var(--fg2);}
  #estudio-raiz .botoes{display:flex;flex-direction:column;gap:9px;}
  #estudio-raiz button.cap{
    width:100%;text-align:left;cursor:pointer;border-radius:10px;
    padding:13px 15px;font-size:16px;font-weight:700;color:#08151a;
    border:none;display:flex;align-items:center;gap:11px;
    transition:transform .1s ease, filter .15s ease;
  }
  #estudio-raiz button.cap:hover{filter:brightness(1.12);}
  #estudio-raiz button.cap:active{transform:scale(.985);}
  #estudio-raiz button.cap .tec{
    font-family:ui-monospace,Menlo,monospace;font-size:12px;font-weight:700;
    background:rgba(0,0,0,.22);border-radius:5px;padding:2px 7px;
  }
  #estudio-raiz button.cap .num{margin-left:auto;font-variant-numeric:tabular-nums;font-size:15px;}
  #estudio-raiz .barra{height:5px;border-radius:99px;background:rgba(255,255,255,.1);margin-top:7px;}
  #estudio-raiz .barra i{display:block;height:100%;border-radius:99px;width:0;transition:width .3s ease;}

  #estudio-raiz .opts{display:flex;flex-wrap:wrap;align-items:center;gap:10px;
    font-size:13px;color:var(--fg2);background:var(--bg2);border-radius:10px;padding:10px 12px;}
  #estudio-raiz .opts input[type=number]{width:62px;background:#0b1418;color:var(--fg);
    border:1px solid var(--linha);border-radius:6px;padding:4px 7px;font-size:13px;}
  #estudio-raiz .opts label{display:flex;align-items:center;gap:6px;cursor:pointer;}

  #estudio-raiz .galeria{display:flex;gap:6px;flex-wrap:wrap;margin-top:14px;
    padding-top:12px;border-top:1px solid var(--linha);min-height:54px;}
  #estudio-raiz .galeria img{width:66px;height:50px;object-fit:cover;border-radius:6px;
    border:2px solid transparent;}
  #estudio-raiz .parar{background:none;border:1px solid var(--linha);color:var(--fg2);
    border-radius:9px;padding:9px;cursor:pointer;font-size:13px;}
  #estudio-raiz .parar:hover{border-color:#ff5c7a;color:#ff5c7a;}
</style>

  <div class="hdr">
    <div class="tit">Estúdio de coleta</div>
    <div class="sub">a câmera vira dataset</div>
    <div class="stat" id="est-stat">abrindo a câmera…</div>
  </div>

  <div class="grid">
    <div class="cam">
      <video id="est-video" autoplay playsinline muted></video>
      <div class="flash" id="est-flash"></div>
      <div class="regressiva" id="est-reg"></div>
      <div class="hud" id="est-hud">pronto</div>
    </div>

    <div class="ctrl">
      <div>
        <div class="lbl">capturar como</div>
        <div class="botoes" id="est-botoes"></div>
      </div>

      <div class="opts">
        <label><input type="checkbox" id="est-rajada" checked> rajada de</label>
        <input type="number" id="est-n" value="12" min="1" max="60">
        <span>fotos, a cada</span>
        <input type="number" id="est-ms" value="200" min="80" max="2000" step="20">
        <span>ms</span>
        <label style="width:100%"><input type="checkbox" id="est-conta"> contagem de 3 antes (para eu me posicionar)</label>
      </div>

      <button class="parar" id="est-parar">encerrar a câmera</button>
    </div>
  </div>

  <div class="galeria" id="est-galeria"></div>
</div>

<script>
(function(){
  const CFG = __CFG__;
  const raiz = document.getElementById('estudio-raiz');
  const video = document.getElementById('est-video');
  const stat  = document.getElementById('est-stat');
  const hud   = document.getElementById('est-hud');
  const flash = document.getElementById('est-flash');
  const reg   = document.getElementById('est-reg');
  const gal   = document.getElementById('est-galeria');
  const box   = document.getElementById('est-botoes');
  let stream = null, ocupado = false;

  const tela = document.createElement('canvas');
  tela.width = CFG.larg; tela.height = CFG.alt;

  /* ── botões, um por classe ── */
  CFG.classes.forEach(function(c){
    const b = document.createElement('button');
    b.className = 'cap';
    b.style.background = c.cor;
    b.dataset.id = c.id;
    b.innerHTML =
      '<span class="tec">' + c.tecla + '</span>' +
      '<span>' + c.rotulo + '</span>' +
      '<span class="num" id="num-' + c.id + '">0</span>';
    const barra = document.createElement('div');
    barra.className = 'barra';
    barra.innerHTML = '<i id="bar-' + c.id + '" style="background:' + c.cor + '"></i>';
    const env = document.createElement('div');
    env.appendChild(b); env.appendChild(barra);
    box.appendChild(env);
    b.onclick = function(){ disparar(c); };
  });

  function pintar(dados){
    if(!dados || !dados.contagem) return;
    let total = 0;
    CFG.classes.forEach(function(c){
      const n = dados.contagem[c.id] || 0;
      total += n;
      const el = document.getElementById('num-' + c.id);
      if (el) el.textContent = n;
      const bar = document.getElementById('bar-' + c.id);
      if (bar) bar.style.width = Math.min(100, 100 * n / (dados.meta || 60)) + '%';
    });
    stat.textContent = total + ' imagens · meta de ' + (dados.meta||60) + ' por classe';
  }
  pintar({contagem: CFG.inicial, meta: CFG.meta});

  /* ── uma captura ── */
  async function capturar(c){
    tela.getContext('2d').drawImage(video, 0, 0, tela.width, tela.height);
    const url = tela.toDataURL('image/jpeg', 0.85);
    flash.classList.add('on');
    setTimeout(function(){ flash.classList.remove('on'); }, 60);

    const mini = document.createElement('img');
    mini.src = url; mini.style.borderColor = c.cor;
    gal.insertBefore(mini, gal.firstChild);
    while (gal.children.length > 12) gal.removeChild(gal.lastChild);

    try{
      const r = await google.colab.kernel.invokeFunction(
        'estudio.salvar', [c.id, url], {});
      pintar(r.data['application/json']);
    }catch(e){
      stat.textContent = 'erro ao salvar: ' + e;
    }
  }

  function espera(ms){ return new Promise(function(r){ setTimeout(r, ms); }); }

  async function disparar(c){
    if (ocupado || !stream) return;
    ocupado = true;
    try{
      if (document.getElementById('est-conta').checked){
        reg.classList.add('on');
        for (let i = 3; i > 0; i--){ reg.textContent = i; await espera(700); }
        reg.classList.remove('on');
      }
      const rajada = document.getElementById('est-rajada').checked;
      const n  = rajada ? Math.max(1, parseInt(document.getElementById('est-n').value || '1', 10)) : 1;
      const ms = Math.max(60, parseInt(document.getElementById('est-ms').value || '200', 10));
      for (let i = 0; i < n; i++){
        hud.textContent = c.rotulo + ' · ' + (i + 1) + ' de ' + n;
        await capturar(c);
        if (i < n - 1) await espera(ms);
      }
      hud.textContent = 'pronto';
    } finally { ocupado = false; }
  }

  /* ── teclas 1..4, o jeito de capturar sem soltar o objeto ── */
  document.addEventListener('keydown', function(ev){
    const alvo = CFG.classes.filter(function(c){ return c.tecla === ev.key; })[0];
    if (alvo && stream) { ev.preventDefault(); disparar(alvo); }
  });

  document.getElementById('est-parar').onclick = function(){
    if (stream) stream.getTracks().forEach(function(t){ t.stop(); });
    stream = null;
    stat.textContent = 'câmera encerrada · o material já está salvo';
    hud.textContent = 'encerrado';
  };

  (async function(){
    try{
      stream = await navigator.mediaDevices.getUserMedia(
        {video: {width: CFG.larg, height: CFG.alt}});
      video.srcObject = stream;
      await video.play();
      stat.textContent = 'câmera pronta · clique no botão ou use as teclas';
      if (window.google && google.colab && google.colab.output)
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight + 40, true);
    }catch(e){
      stat.textContent = 'a câmera não abriu: ' + e;
      hud.textContent = 'sem câmera';
    }
  })();
})();
</script>
"""

In [ ]:
# ── as classes: mude aqui se quiser outro par ──────────────────────
CLASSES = ["com_epi", "sem_epi"]
META    = 60          # alvo por classe · 40 ja treina, 60 e confortavel

estudio = Estudio(raiz=f"{DRIVE}/05-epi", classes=CLASSES, meta=META)
estudio.abrir()

In [ ]:
# Rode esta célula quando terminar de coletar, para conferir o que tem.
estudio.resumo()

## Passo 2 · as duas pastas

Este é o momento do slide. Nenhuma regra foi escrita, nenhuma caixa foi
desenhada: as fotos foram **separadas em duas pastas**, e o nome da pasta é o
gabarito.

A parte de `prova` não é burocracia. É a única forma de saber se ele **aprendeu**
ou se **decorou** — e as fotos dela não entram no treino.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DAS FOTOS PARA AS DUAS PASTAS
#
# Este é o momento do slide: "eu não escrevi nenhuma regra. Eu separei
# as fotos em duas pastas, e o nome da pasta é o gabarito."
#
# A divisão entre treino e prova NÃO é burocracia: é a única forma de
# saber se o modelo aprendeu ou decorou. Se as mesmas fotos entram nos
# dois lados, o número na tela dá quase 100% e não significa nada.
#
# A semente é fixa para o ensaio e o palco darem o mesmo resultado.
# ═══════════════════════════════════════════════════════════════
import os, glob, random, shutil


def preparar_dataset(estudio, val_frac=0.2, semente=7, minimo=12):
    """Embaralha o material bruto e monta treino/ e val/ do YOLO."""
    destino = os.path.join(estudio.raiz, "treino")
    shutil.rmtree(destino, ignore_errors=True)

    random.seed(semente)
    linhas = []
    for classe in estudio.classes:
        fotos = estudio.arquivos(classe)
        if len(fotos) < minimo:
            # ⚠️ o operador de formatação NÃO pode abrir a linha aqui. Uma linha
            # que começa com % (ou !) é lida como magia de notebook, e o
            # tools/validar_colabs.py reprova a célula inteira por isso. Por
            # segurança de palco, o formato vai por .format() mesmo.
            raise SystemExit(
                "a classe '{}' tem só {} imagens. Volte ao estúdio e colete "
                "mais: abaixo de {} o treino não tem o que aprender.".format(
                    classe, len(fotos), minimo))
        random.shuffle(fotos)
        corte = max(3, int(len(fotos) * val_frac))
        partes = {"val": fotos[:corte], "train": fotos[corte:]}
        for parte, arquivos in partes.items():
            pasta = os.path.join(destino, parte, classe)
            os.makedirs(pasta, exist_ok=True)
            for f in arquivos:
                shutil.copy(f, os.path.join(pasta, os.path.basename(f)))
        linhas.append((classe, len(partes["train"]), len(partes["val"])))

    print("dataset montado em:", destino)
    print()
    print("  %-16s %8s %8s" % ("classe", "treino", "prova"))
    for c, tr, va in linhas:
        print("  %-16s %8d %8d" % (c, tr, va))
    print()
    print("as fotos da coluna 'prova' NÃO entram no treino: é com elas que")
    print("a gente descobre se ele aprendeu ou se só decorou.")
    return destino


def amostra_do_dataset(estudio, por_classe=4):
    """Mostra na tela algumas fotos de cada pasta — a prova visual."""
    import matplotlib.pyplot as plt, cv2
    n = len(estudio.classes)
    fig, axs = plt.subplots(n, por_classe, figsize=(5.2 * por_classe, 4.4 * n))
    axs = axs.reshape(n, por_classe)
    for i, classe in enumerate(estudio.classes):
        fotos = estudio.arquivos(classe)[:por_classe]
        for j in range(por_classe):
            ax = axs[i][j]; ax.axis("off")
            if j < len(fotos):
                ax.imshow(cv2.cvtColor(cv2.imread(fotos[j]), cv2.COLOR_BGR2RGB))
            if j == 0:
                ax.set_title(estudio.rotulos[i].upper(), fontsize=30,
                             loc="left", color=VERDE if i == 0 else VERMELHO)
    plt.tight_layout(); plt.show()

In [ ]:
PASTA = preparar_dataset(estudio, val_frac=0.2)
amostra_do_dataset(estudio, por_classe=4)

## Passo 3 · o treino, acontecendo na tela

**Ajuste `EPOCAS` na frente da plateia.** Rodar com 5 e depois com 25 é a melhor
demonstração possível de "por que treinar mais importa": o contraste entre as
duas curvas dispensa qualquer explicação técnica.

Enquanto roda, narre: *"essa linha vermelha é o erro dele. Cada época é uma
passada por todas as fotos. Olhem: ele está errando menos. Ninguém corrigiu à
mão. Ele ajusta sozinho, um tantinho por vez, na direção de errar menos."*

In [ ]:
from IPython.display import clear_output
import matplotlib.pyplot as plt

def treinar_mostrando(modelo, dados, epocas, imgsz=224, batch=32,
                      projeto="/content/runs", nome="ao_vivo", titulo="Aprendendo"):
    """Treina e redesenha a curva a cada epoca — o ponto alto da demo."""
    hist = {}                                  # epoca -> (perda, acuracia)

    def a_cada_epoca(trainer):
        m = getattr(trainer, "metrics", None) or {}
        acc = m.get("metrics/accuracy_top1")
        perda = float(trainer.loss.item()) if getattr(trainer, "loss", None) is not None else None
        hist[trainer.epoch + 1] = (perda, acc)   # dict: a epoca repetida sobrescreve

        eps = sorted(hist)
        perdas = [hist[e][0] for e in eps]
        accs = [(hist[e][1] or 0) * 100 for e in eps]

        clear_output(wait=True)
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(20, 8))
        a1.plot(eps, perdas, lw=5, color="#ff5c5c", marker="o", ms=10)
        a1.set_title("ERRO — tem que descer"); a1.set_xlabel("época")
        a2.plot(eps, accs, lw=5, color="#3fe0a8", marker="o", ms=10)
        a2.set_ylim(0, 101)
        a2.set_title("ACERTO — tem que subir"); a2.set_xlabel("época"); a2.set_ylabel("%")
        if accs:
            a2.text(eps[-1], accs[-1], f"  {accs[-1]:.0f}%", fontsize=34,
                    color="#3fe0a8", va="center", fontweight="bold")
        fig.suptitle(f"{titulo} · época {max(eps)} de {epocas}", fontsize=34)
        plt.tight_layout(); plt.show()

    modelo.add_callback("on_fit_epoch_end", a_cada_epoca)
    r = modelo.train(data=dados, epochs=epocas, imgsz=imgsz, batch=batch,
                     project=projeto, name=nome, exist_ok=True, verbose=False, plots=True)
    print("pesos e graficos em:", r.save_dir)
    return r

In [ ]:
EPOCAS = 25          # ← o botao do palco

modelo = YOLO(f"{DRIVE}/00-pesos/yolo11n-cls.pt")
res = treinar_mostrando(
    modelo,
    dados=PASTA,
    epocas=EPOCAS,
    titulo="Aprendendo a reconhecer EPI",
    nome="epi",
)

In [ ]:
import shutil, os
os.makedirs(f"{DRIVE}/05-epi/pesos", exist_ok=True)
destino = f"{DRIVE}/05-epi/pesos/epi_best.pt"
shutil.copy(f"{res.save_dir}/weights/best.pt", destino)
print("especialista salvo em:", destino)
print("no proximo evento da para carregar direto daqui, sem treinar de novo.")

## Passo 4 · a prova, só com o que ele nunca viu

Aqui não interessa o número redondo. Interessa **que tipo** de erro ele comete:
dizer "tem capacete" quando não tem é muito pior que o contrário, e nenhuma
acurácia sozinha conta isso.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# A PROVA — só com o que ele nunca viu
#
# O slide chama isso de quarto passo da receita, e é o que separa
# aprender de decorar. Aqui não interessa o número redondo: interessa
# QUAL erro ele comete. Dizer "tem capacete" quando não tem é muito
# pior que o contrário, e nenhuma acurácia sozinha conta isso.
# ═══════════════════════════════════════════════════════════════
import glob, os, random
import matplotlib.pyplot as plt, cv2


def provar(modelo, pasta_treino, quantas=8, semente=3):
    prova = []
    for classe in sorted(os.listdir(os.path.join(pasta_treino, "val"))):
        for f in glob.glob(os.path.join(pasta_treino, "val", classe, "*")):
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                prova.append((f, classe))
    if not prova:
        raise SystemExit("não há fotos de prova — rode preparar_dataset antes")

    random.seed(semente)
    random.shuffle(prova)
    prova = prova[:quantas]

    colunas = 4
    linhas = (len(prova) + colunas - 1) // colunas
    fig, axs = plt.subplots(linhas, colunas, figsize=(5.5 * colunas, 4.6 * linhas))
    axs = axs.reshape(linhas, colunas)
    acertos = 0
    for k in range(linhas * colunas):
        ax = axs[k // colunas][k % colunas]; ax.axis("off")
        if k >= len(prova):
            continue
        arquivo, gabarito = prova[k]
        p = modelo.predict(arquivo, verbose=False)[0]
        palpite = p.names[int(p.probs.top1)]
        certeza = float(p.probs.top1conf)
        ok = (palpite == gabarito)
        acertos += ok
        ax.imshow(cv2.cvtColor(cv2.imread(arquivo), cv2.COLOR_BGR2RGB))
        ax.set_title("%s  %.0f%%" % (palpite, certeza * 100), fontsize=24,
                     color=VERDE if ok else VERMELHO)
    fig.suptitle("Prova final: %d de %d corretas" % (acertos, len(prova)), fontsize=36)
    plt.tight_layout(); plt.show()
    return acertos, len(prova)

In [ ]:
especialista = YOLO(f"{DRIVE}/05-epi/pesos/epi_best.pt")
provar(especialista, PASTA, quantas=8)

## Passo 5 · onde ele ainda erra

A matriz de confusão é o slide honesto da IA. Ela mostra **para que lado** o
modelo erra, que é a informação que decide se dá para colocar isso em produção.

In [ ]:
from IPython.display import Image, display
import os
for nome in ["confusion_matrix_normalized.png", "confusion_matrix.png", "results.png"]:
    caminho = os.path.join(str(res.save_dir), nome)
    if os.path.exists(caminho):
        print(nome)
        display(Image(filename=caminho, width=1100))

## Passo 6 · ao vivo, na câmera

O fecho da demonstração: aponte a câmera para você e **tire o capacete no ar**.
O painel vira na frente da plateia.

Encerre pelo botão verde, e não interrompendo a célula: assim a câmera é
liberada direito e a próxima demo abre sem reclamar.

In [ ]:
# ── motor de webcam ao vivo (leia o comentário: é o truque da demo) ──
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2, numpy as np, PIL.Image, io, time

def iniciar_webcam(largura=640, altura=480):
    # cria o video no navegador + a camada de overlay por cima dele
    display(Javascript('''
      var video, div = null, stream, imgElement, labelElement, captureCanvas;
      var pendingResolve = null, shutdown = false;
      var LARG = %d, ALT = %d;

      function removeDom() {
        if (stream) stream.getVideoTracks()[0].stop();
        if (video) video.remove();
        if (div) div.remove();
        video = null; div = null; stream = null;
        imgElement = null; captureCanvas = null; labelElement = null;
      }

      function onAnimationFrame() {
        if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
        if (pendingResolve) {
          var result = "";
          if (!shutdown) {
            captureCanvas.getContext('2d').drawImage(video, 0, 0, LARG, ALT);
            result = captureCanvas.toDataURL('image/jpeg', 0.75);
          }
          var lp = pendingResolve;
          pendingResolve = null;
          lp(result);
        }
      }

      async function criarDom() {
        if (div !== null) return stream;

        div = document.createElement('div');
        div.style.border = '2px solid #3fe0a8';
        div.style.padding = '3px';
        div.style.width = '100%%';
        div.style.maxWidth = '900px';
        div.style.borderRadius = '10px';
        document.body.appendChild(div);

        var parar = document.createElement('div');
        parar.innerHTML = '&#9632; clique aqui para encerrar';
        parar.style.cssText = 'cursor:pointer;background:#3fe0a8;color:#06231a;' +
          'font-weight:700;padding:10px 16px;border-radius:8px;text-align:center;' +
          'font-family:system-ui,sans-serif;font-size:18px';
        div.appendChild(parar);
        parar.onclick = function() { shutdown = true; };

        video = document.createElement('video');
        video.style.display = 'block';
        video.style.width = '100%%';
        video.setAttribute('playsinline', '');
        video.onclick = function() { shutdown = true; };

        stream = await navigator.mediaDevices.getUserMedia(
          {video: {width: LARG, height: ALT}});
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        // a camada que recebe o resultado do modelo, por cima do vídeo
        imgElement = document.createElement('img');
        imgElement.style.position = 'absolute';
        imgElement.style.zIndex = 1;
        imgElement.style.pointerEvents = 'none';
        imgElement.onclick = function() { shutdown = true; };
        div.appendChild(imgElement);

        labelElement = document.createElement('div');
        labelElement.style.cssText = 'font-family:system-ui,sans-serif;' +
          'font-size:20px;color:#e6edf3;padding:8px 4px';
        div.appendChild(labelElement);

        captureCanvas = document.createElement('canvas');
        captureCanvas.width = LARG;
        captureCanvas.height = ALT;
        window.requestAnimationFrame(onAnimationFrame);

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        return stream;
      }

      async function quadro(rotulo, overlay) {
        if (shutdown) { removeDom(); shutdown = false; return ''; }
        stream = await criarDom();
        if (rotulo != "") labelElement.innerHTML = rotulo;
        if (overlay != "") {
          var r = video.getClientRects()[0];
          imgElement.style.top = r.top + "px";
          imgElement.style.left = r.left + "px";
          imgElement.style.width = r.width + "px";
          imgElement.style.height = r.height + "px";
          imgElement.src = overlay;
        }
        var result = await new Promise(function(resolve) { pendingResolve = resolve; });
        shutdown = false;
        return {'img': result};
      }
    ''' % (largura, altura)))


def _para_imagem(resposta):
    # base64 do navegador -> imagem BGR do OpenCV
    if not resposta:
        return None
    dados = b64decode(resposta.split(',')[1])
    arr = np.frombuffer(dados, dtype=np.uint8)
    return cv2.imdecode(arr, flags=1)


def _para_overlay(rgba):
    # array RGBA -> data URI PNG, para o navegador sobrepor ao video
    img = PIL.Image.fromarray(rgba, 'RGBA')
    buf = io.BytesIO()
    img.save(buf, format='png')
    return 'data:image/png;base64,' + b64encode(buf.getvalue()).decode('utf-8')


def rodar_ao_vivo(processa, largura=640, altura=480, rotulo_inicial='iniciando…'):
    # Laco principal.
    #
    # `processa(frame_bgr, overlay_rgba)` recebe o quadro e uma tela RGBA
    # transparente do mesmo tamanho, desenha nela, e devolve o texto do
    # painel. Encerre clicando no botão verde (ou no próprio vídeo).
    iniciar_webcam(largura, altura)
    overlay = np.zeros([altura, largura, 4], dtype=np.uint8)
    envio = ''
    rotulo = rotulo_inicial
    n = 0
    t0 = time.time()
    try:
        while True:
            resposta = eval_js('quadro("{}", "{}")'.format(rotulo, envio))
            if not resposta:
                break
            frame = _para_imagem(resposta['img'])
            if frame is None:
                break

            overlay[:] = 0
            texto = processa(frame, overlay)

            n += 1
            fps = n / max(1e-6, time.time() - t0)
            rotulo = '{} &nbsp;·&nbsp; {:.1f} quadros/s'.format(texto, fps)
            envio = _para_overlay(overlay)
    except Exception as e:
        print('encerrado:', type(e).__name__, e)
    print('fim · {} quadros processados'.format(n))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# INFERÊNCIA AO VIVO — o modelo que acabou de nascer, decidindo
#
# O vídeo NUNCA sai do navegador: ele toca nativo, a 30 quadros por
# segundo, e só um PNG transparente com o painel volta do servidor.
# É a diferença entre parecer travado e parecer mágica.
#
# O painel é grande de propósito: a plateia está a 6-10 m do telão, e
# rótulo em corpo de biblioteca de visão computacional some ali.
# ═══════════════════════════════════════════════════════════════
import cv2, numpy as np

FONTE = cv2.FONT_HERSHEY_SIMPLEX


def painel_classificacao(modelo, cores=None, limiar=0.60):
    """Devolve um `processa` pronto para o rodar_ao_vivo: classifica o
    quadro inteiro e desenha o veredito em tamanho de palco.

    cores: {"nome_da_classe": (R, G, B)} — ordem RGB, ver o aviso do topo.
    limiar: abaixo disso o painel assume "em duvida" em vez de cravar."""
    cores = cores or {}

    def processa(frame, overlay):
        r = modelo.predict(frame, verbose=False, imgsz=224)[0]
        nome = r.names[int(r.probs.top1)]
        conf = float(r.probs.top1conf)
        h, w = overlay.shape[:2]

        cor = tuple(cores.get(nome, (233, 242, 239)))
        seguro = conf >= limiar

        # faixa superior: o veredito, legível do fundo da sala
        cv2.rectangle(overlay, (0, 0), (w, 84), (11, 20, 24, 225), -1)
        cv2.putText(overlay, nome.upper().replace("_", " "), (18, 57),
                    FONTE, 1.6, cor + (255,), 4, cv2.LINE_AA)
        cv2.putText(overlay, "%.0f%%" % (conf * 100), (w - 175, 57),
                    FONTE, 1.4,
                    (255, 255, 255, 255) if seguro else (180, 180, 180, 210),
                    3, cv2.LINE_AA)

        # moldura grossa quando o modelo está seguro, fina quando titubeia
        cv2.rectangle(overlay, (1, 1), (w - 2, h - 2), cor + (255,),
                      10 if seguro else 3)

        # barra de confianca no rodape
        cv2.rectangle(overlay, (0, h - 16), (int(w * conf), h), cor + (255,), -1)

        if not seguro:
            cv2.putText(overlay, "em duvida", (18, h - 32),
                        FONTE, 0.85, (255, 184, 77, 255), 2, cv2.LINE_AA)
        return "%s · %.0f%%" % (nome, conf * 100)

    return processa

In [ ]:
# cor por classe, em (R, G, B) — o overlay e RGBA, ver o comentario do bloco
CORES = {
    "com_epi": (63, 224, 168),      # verde: pode entrar
    "sem_epi": (255, 92, 122),      # vermelho: barra
}

rodar_ao_vivo(painel_classificacao(especialista, cores=CORES, limiar=0.60),
              rotulo_inicial="olhando…")

---

## Se a internet cair no evento

Tudo que este notebook produz fica no seu Drive assim que roda:
`05-epi/_bruto/` com as fotos, `05-epi/treino/` com as pastas e
`05-epi/pesos/epi_best.pt` com o especialista.

**No ensaio, grave a tela.** Guarde em `99-reserva/`. Se o Colab não abrir na
hora, você mostra a gravação e conta exatamente a mesma história, sem pedir
desculpa à plateia.

**Se a câmera não abrir** na máquina do evento, o `foto()` do Passo 0 continua
funcionando: dá para montar um dataset pequeno com uma foto de cada vez. Menos
impressionante, e nunca falha.

**Se a GPU não vier**, o treino roda em CPU: com 224 pixels e 25 épocas leva
alguns minutos em vez de alguns segundos. Rode a checagem do Passo 0 antes de
subir ao palco e, se não houver GPU, baixe `EPOCAS` para 10.